# YuvaIntern Week 1 – Data Acquisition, Cleaning and Preprocessing

**Dataset:** Breast Cancer Wisconsin (Diagnostic) Dataset  
**Libraries:** pandas, numpy, scikit-learn, matplotlib

This notebook documents the complete Week 1 preprocessing workflow.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

## 1. Data Acquisition
The public Breast Cancer Wisconsin (Diagnostic) dataset is loaded reproducibly using scikit-learn.

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
df["diagnosis"] = df["target"].map({0: "malignant", 1: "benign"})

print("Dataset shape:", df.shape)
df.head()

## 2. Initial Data Exploration
Check data types, missing values, duplicate rows, and basic statistics.

In [ ]:
print(df.info())
print("\nMissing values:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df.describe().T.head(10)

## 3. Missing-Value Analysis
No missing values are present, so no artificial imputation is applied.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False).head(10)
missing.plot(kind="bar", figsize=(10,4), title="Missing Values – Top Columns")
plt.ylabel("Missing Count")
plt.tight_layout()
plt.show()

## 4. Duplicate Check and Cleaning

In [ ]:
clean = df.drop_duplicates().copy()
print("Rows before:", len(df))
print("Rows after:", len(clean))

## 5. Outlier Detection Using IQR
Potential outliers are identified with the 1.5×IQR rule.

In [ ]:
feature_cols = list(data.feature_names)
outlier_counts = {}

for col in feature_cols:
    q1, q3 = clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_counts[col] = int(((clean[col] < lower) | (clean[col] > upper)).sum())

print(pd.Series(outlier_counts).sort_values(ascending=False).head(10))

In [ ]:
top_outliers = pd.Series(outlier_counts).sort_values(ascending=False).head(10)
top_outliers.plot(kind="bar", figsize=(10,4), title="Top 10 Features by IQR Outlier Count")
plt.ylabel("Number of Outlier Observations")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Outlier Treatment
Extreme values are capped at the IQR limits rather than deleting complete observations.

In [ ]:
for col in feature_cols:
    q1, q3 = clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    clean[col] = clean[col].clip(lower=lower, upper=upper)

## 7. Feature Scaling
StandardScaler is used to put numerical predictors on a comparable scale.

In [ ]:
scaler = StandardScaler()
clean[feature_cols] = scaler.fit_transform(clean[feature_cols])
clean[feature_cols[:8]].boxplot(figsize=(10,5), rot=45)
plt.title("Standardized Numerical Features After Preprocessing")
plt.ylabel("Standardized Value")
plt.tight_layout()
plt.show()

## 8. Final Dataset Check and Export

In [ ]:
print("Final shape:", clean.shape)
print("Remaining missing values:", int(clean.isna().sum().sum()))
print(clean.head())
clean.to_csv("YuvaIntern_Week1_Breast_Cancer_Cleaned.csv", index=False)
print("Saved cleaned dataset.")

## 9. Conclusion
The dataset was checked for missing values and duplicates, potential outliers were treated using IQR capping, and numerical features were standardized. The processed dataset is ready for further exploratory analysis and machine-learning tasks.